# 13주 · 다항 회귀·보간·스플라인

수치해석및실습 CE273 · 2027학년도 1학기

원본 슬라이드의 개념·예제·차시 순서에 연결한 **신규 병행 실습**. 원본 코드의 단순 복사본과 구분

**목표**: 수치 원리·도구 기능 이해 → 좋은 AI 질문·지시 → 오차·수렴·결과의 검토

**수업**: 2027-05-25 곡선 적합 (2) · 대면 / 2027-05-27 곡선 적합 (3) · 대면

[Excel 실습](https://raw.githubusercontent.com/steelpl/lecture/main/CE273-numerical-analysis/week13/CE273_W13.xlsx) · [주차 안내](https://github.com/steelpl/lecture/tree/main/CE273-numerical-analysis/week13)

슬라이드 예측 → 작은 손계산 → 실행 → 한 조건 변경 → AI 요청 수정 → 단위·오차·수렴·실패 상태 검토

셀 실행: Shift+Enter. 처음에는 위에서 아래로 실행. 공통 준비 셀 실행 후 각 절의 개별 실행도 가능. None 항목과 변경 셀은 직접 작성

## 0. 실행 환경·보조 함수

필요한 버전만 설치. 업데이트한 패키지를 이미 불러온 경우 런타임 재시작 후 처음부터 실행

보조 함수의 세부 문법 암기 불필요. 입력·처리·반환값·오차 기준·실패 조건부터 확인. 아래 실습의 호출 코드와 슬라이드 식 대조

In [ ]:
# 최초 실행 시 필요한 버전만 설치. Colab에서 위에서 아래로 실행
import importlib.metadata as metadata, subprocess, sys
from packaging.version import Version
requirements = {"numpy": "1.26", "pandas": "2.2", "scipy": "1.11", "matplotlib": "3.8", "openpyxl": "3.1"}
for package, minimum in requirements.items():
    try:
        outdated = Version(metadata.version(package)) < Version(minimum)
    except metadata.PackageNotFoundError:
        outdated = True
    if outdated:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", f"{package}>={minimum}"])
print("실행 환경 준비 완료. 이미 불러온 패키지를 업데이트한 경우 런타임 재시작 후 처음부터 실행")

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import optimize, linalg, interpolate, integrate

def vector(values, minimum=1):
    x = np.asarray(values, dtype=float)
    if x.ndim != 1 or len(x) < minimum or not np.isfinite(x).all():
        raise ValueError("유한한 1차원 자료·최소 원소 수 확인 필요")
    return x

def relative_error(truth, estimate):
    # 기준값 0에서 상대오차를 정의하지 않음. 절대오차 별도 확인
    return np.nan if truth == 0 else abs((truth-estimate)/truth)


## S01. 원본 6쌍의 2차 회귀

**슬라이드 연결**: 2026_S1_NA_Wk13a_CurveFit2.pptx p9, 10, 11

**자료 구분**: 원본 예제·수치·원리에 기반. 코드에 표시한 추가 검토는 조건 변경 실험

**먼저 예측**: 차수를 높이면 새로운 자료의 예측도 항상 좋아지는가?

단위·스텝·초기값·분모·조건 중 이번 예제의 핵심을 실행 전에 기록

In [ ]:
x=np.arange(6,dtype=float);y=np.array([2.1,7.7,13.6,27.2,40.9,61.1])
X=np.vander(x,3,increasing=True)
coef=np.linalg.lstsq(X,y,rcond=None)[0];r=y-X@coef
print("a0,a1,a2:",coef,"SSE:",r@r,"condition(X):",np.linalg.cond(X),"condition(XTX):",np.linalg.cond(X.T@X))
assert np.allclose(coef,[2.47857142857143,2.35928571428571,1.86071428571429])
assert np.isclose(r@r,3.74657142857143)
print("원본 Excel은 계수를 5자리 정도로 고정. 새 파일은 입력에서 계수 재계산")

**조건 변경**: 최고차수·x의 중심화·스케일링을 바꾸고 조건수 비교

원본 셀을 복사한 뒤 한 조건씩 변경. 기준 결과와 변경 결과를 구분

**AI 요청에 넣을 정보**: 원본6쌍·차수2·계수 순서·rank·SSE·수치적 안정성·과적합 검토 명시

**결과 검토**: 정규방정식은 유도용. 계산은 QR/SVD 기반 lstsq 우선

예상·실행·독립 검산·수정 지시·채택 또는 거절 이유 기록

In [ ]:
# 바꾼 조건과 검산 코드를 이 셀에 추가


## S02. 원본 ln2의 Newton·Lagrange 보간

**슬라이드 연결**: 2026_S1_NA_Wk13a_CurveFit2.pptx p24, 31, 36 / 2026_S1_NA_Wk13b_CurveFit3.pptx p7

**자료 구분**: 원본 예제·수치·원리에 기반. 코드에 표시한 추가 검토는 조건 변경 실험

**먼저 예측**: 같은 점을 쓴 Newton·Lagrange 보간은 다른 함수를 만드는가?

단위·스텝·초기값·분모·조건 중 이번 예제의 핵심을 실행 전에 기록

In [ ]:
x=np.array([1.,4,6]);y=np.array([0.,1.386294,1.791759])
lagrange=interpolate.BarycentricInterpolator(x,y)
b1=(y[1]-y[0])/(x[1]-x[0]);b2=((y[2]-y[1])/(x[2]-x[1])-b1)/(x[2]-x[0])
newton=y[0]+b1*(2-x[0])+b2*(2-x[0])*(2-x[1])
linear_16=np.interp(2,x[[0,2]],y[[0,2]]);linear_14=np.interp(2,x[:2],y[:2])
print({"linear_1_6":linear_16,"linear_1_4":linear_14,"quadratic":newton,"true_ln2":np.log(2)})
assert np.isclose(newton,lagrange(2)) and np.isclose(newton,.5658442)
print("두 방법에 같은 반올림 자료 사용. 원본 페이지의 ln6 끝자리 1 차이는 별도 구분")

**조건 변경**: 세 번째 점을 추가한 효과와 반올림 자릿수 변경 효과 구분

원본 셀을 복사한 뒤 한 조건씩 변경. 기준 결과와 변경 결과를 구분

**AI 요청에 넣을 정보**: 서로 다른 x·원본 반올림 값·보간 차수·평가점2·외삽 금지·방법 대조 요청

**결과 검토**: 보간점에서는 일치해도 구간 밖 정확성은 보장되지 않음

예상·실행·독립 검산·수정 지시·채택 또는 거절 이유 기록

In [ ]:
# 바꾼 조건과 검산 코드를 이 셀에 추가


## S03. 원본 1차 스플라인과 경계 조건

**슬라이드 연결**: 2026_S1_NA_Wk13b_CurveFit3.pptx p15, 16, 18

**자료 구분**: 원본 예제·수치·원리에 기반. 코드에 표시한 추가 검토는 조건 변경 실험

**먼저 예측**: 함수값이 이어진다는 것과 기울기가 이어진다는 것은 같은가?

단위·스텝·초기값·분모·조건 중 이번 예제의 핵심을 실행 전에 기록

In [ ]:
x=np.array([3.,4.5,7,9]);y=np.array([2.5,1,2.5,.5])
linear=np.interp(5,x,y);natural=interpolate.CubicSpline(x,y,bc_type="natural")
print("원본 1차 스플라인 x5:",linear,"자연 3차 스플라인:",natural(5))
assert np.isclose(linear,1.3) and np.allclose(natural(x),y)
print("3차 스플라인 경계 조건은 추가 모형 선택. 원본 선형 결과와 같을 필요 없음")

**조건 변경**: natural과 not-a-knot 경계 조건의 차이를 설명한 뒤 비교

원본 셀을 복사한 뒤 한 조건씩 변경. 기준 결과와 변경 결과를 구분

**AI 요청에 넣을 정보**: 원본4점·구간·스플라인 차수·경계 조건·보간점 일치 검산 명시

**결과 검토**: 라이브러리 기본 경계 조건을 숨기지 않음

예상·실행·독립 검산·수정 지시·채택 또는 거절 이유 기록

In [ ]:
# 바꾼 조건과 검산 코드를 이 셀에 추가


## 주차 확인

회귀·보간·경계 조건·같은 자료의 비교 기록

입력 항목:

- `2차회귀_a2`: 해당 계산값·항 수·오차 기준의 수치
- `ln2_2차보간`: 해당 계산값·항 수·오차 기준의 수치
- `1차스플라인_x5`: 해당 계산값·항 수·오차 기준의 수치
- `보간과_회귀는_항상_같음`: 판단 · Python `True`/`False`

공개 계산·개념 확인용 항목. 값만 복사하면 학습 완료로 판단 불가. 가정·적용 범위·검산 근거는 아래 서술과 함께 확인

In [ ]:
answers = {'2차회귀_a2': None, 'ln2_2차보간': None, '1차스플라인_x5': None, '보간과_회귀는_항상_같음': None}


In [ ]:
def check_answers(values):
    expected = {'2차회귀_a2': 1.86071428571429, 'ln2_2차보간': 0.5658442, '1차스플라인_x5': 1.3, '보간과_회귀는_항상_같음': False}
    pending, wrong = [], []
    for key, target in expected.items():
        got = values.get(key)
        if got is None:
            pending.append(key)
        elif isinstance(target, bool):
            if not isinstance(got, (bool, np.bool_)) or bool(got) != target:
                wrong.append(key)
        elif isinstance(target, str):
            if not isinstance(got, str) or got.strip().lower() != target:
                wrong.append(key)
        elif isinstance(got, (bool, np.bool_)) or not isinstance(got, (int, float, np.number)) or not np.isfinite(got) or not np.isclose(got, target, rtol=1e-6, atol=1e-8):
            wrong.append(key)
    if pending:
        print("미완료:", ", ".join(pending))
    if wrong:
        print("재검토:", ", ".join(wrong))
    if not pending and not wrong:
        print("공개 계산·개념 확인 통과. 해석·가정·AI 검토 기록은 별도 확인 필요")
    return {"pending": pending, "wrong": wrong, "passed": not pending and not wrong}

check_result = check_answers(answers)

## AI 요청·검토 기록

1. 문제·모형·입력·단위·자료 출처
2. 최초 요청: 목적·조건·알고리즘·출력·검증 기준
3. 실행 전 예측·작은 손계산의 근거
4. 의심한 출력·오차·수렴·실패 조건과 수정 지시
5. 채택·수정·거절한 내용, 독립 검산·한계
6. AI 없이 설명 가능한 원리 한 가지

글자 수 자동 채점 없음. 실행 성공·작은 갱신량·작은 잔차와 문제 해결의 타당성 구분

In [ ]:
review_log = {
    "문제·모형·자료·단위": None,
    "최초 AI 요청": None,
    "예측·손계산": None,
    "의심한 부분·수정 지시": None,
    "검산·최종 판단·한계": None,
    "내 설명": None,
}
